### Porcentaje de personas que tardan más de una hora en llegar a sus centros de estudio o trabajo en la Zona Metropolitana de la Ciudad de México (ZMCM 2020)

**Indicador:** Porcentaje de personas residentes en la Zona Metropolitana de la Ciudad de México (delimitación 2020) que tardan más de una hora en trasladarse a sus centros de estudio o trabajo, respecto del total de personas que se trasladan a dichos destinos | **Fórmula:** (PobTrasladoLargo / PobTrabEstTraslado) × 100 | **Fuente:** INEGI, Encuesta Intercensal 2015, Censo de Población y Vivienda 2020 (cuestionario ampliado) y Encuesta Intercensal 2025, microdatos de personas | **Universo:** Personas residentes en los municipios de la ZMCM 2020 (CDMX, Estado de México e Hidalgo) que se trasladan a su centro de estudio y/o trabajo

**Método:** Estimación con diseño muestral complejo (upm, estrato, factor de expansión).
- **Ultimos ajustes:** el Censo/EIC no permite identificar "una hora o más"; las categorías disponibles son "31 minutos a 1 hora" y "más de 1 hora y hasta 2 horas". Por ello el indicador se mide como "**más** de una hora".
- **PobTrasladoLargo:** personas con `tie_traslado_trab` o `tie_traslado_escu` en 4 (más de 1 hora y hasta 2 horas) o 5 (más de 2 horas). Quien estudia y trabaja se cuenta una sola vez si alguno de sus traslados supera la hora.
- **PobTrabEstTraslado:** personas con al menos un traslado a estudio o trabajo con tiempo declarado (códigos 1 a 5). Se excluyen quienes no se trasladan, no es posible determinarlo o no especificado.
- **Universo territorial:** municipios con 1 en la columna `ZMCM_2020` de `ZMVM_2015_2020_2025.xlsx` (16 alcaldías de la CDMX, municipios del Estado de México y de Hidalgo). Se usa la misma delimitación 2020 para los tres años, de modo que la serie compara siempre el mismo territorio.
- Se descargan los microdatos completos de las entidades 09, 13 y 15 y la ZMCM se trata como **dominio** de estimación (`where`), no como subconjunto, para conservar el diseño muestral completo en la varianza. Estratos y UPM se prefijan con la entidad para que no se repitan entre estados.
- No se restringe por entidad de destino: se incluyen residentes de la ZMCM que estudian o trabajan fuera de ella.

**Comparabilidad 2015-2020-2025:** las tres fuentes usan la misma pregunta ("¿Cuánto tiempo hace (NOMBRE) de aquí a su trabajo / escuela?"), los mismos nombres de variables (`TIE_TRASLADO_TRAB`, `TIE_TRASLADO_ESCU`) y los mismos códigos 1-5; los códigos de exclusión también coinciden (trabajo: 6 no es posible determinarlo, 7 no se traslada, 9 no especificado; escuela: 6 no se traslada, 9 no especificado). Única diferencia estructural: en la EIC 2025 la entidad y el municipio se llaman `CVE_ENT`/`CVE_MUN` (antes `ENT`/`MUN`); se homologan al leer. Diccionarios: [EIC 2015](https://www.inegi.org.mx/rnm/index.php/catalog/214/data-dictionary/F18?file_name=TR_Persona), [Censo 2020](https://www.inegi.org.mx/rnm/index.php/catalog/632/data-dictionary/F13?file_name=Personas_CA), [EIC 2025](https://www.inegi.org.mx/contenidos/programas/eic/2025/microdatos/eic2025_micro_fd.xlsx) (hoja PERSONAS).

La prospectiva a 2047 usa la TVMA (tasa de variación media anual) histórica 2015-2025 aplicada con distinto nivel de esfuerzo y dirección por escenario.

In [1]:
# =============================================================================
# BLOQUE 00: LIBRERÍAS
# =============================================================================
import os
import tempfile
import zipfile

import httpx
import pandas as pd
import polars as pl
import svy

pl.Config.set_tbl_rows(-1) 
from IPython.display import display
from rich import print as rprint  

In [3]:
# =============================================================================
# BLOQUE 0: MUNICIPIOS ZMCM 2020 Y DESCARGA DE MICRODATOS (EIC 2015, CENSO 2020 CA, EIC 2025)
# =============================================================================
zm = pd.read_excel("ZMVM_2015_2020_2025.xlsx", dtype=str)
zm = zm[zm["ZMCM_2020"].notna()]
cvegeo_zm = (zm["CVE_ENT"].str.zfill(2) + zm["CVE_MUN"].str.zfill(3)).tolist()
entidades = sorted(zm["CVE_ENT"].str.zfill(2).unique())
print(f"ZMCM 2020: {len(cvegeo_zm)} municipios en entidades {entidades}")
print(zm.groupby("CVE_ENT").size().to_string())

abrev_censo2020 = {"09": "cdmx", "13": "hgo", "15": "mex"}
urls_censo_int = {
    2015: "https://www.inegi.org.mx/contenidos/programas/intercensal/2015/microdatos/eic2015_{ent}_csv.zip",
    2020: "https://www.inegi.org.mx/contenidos/programas/ccpv/2020/microdatos/Censo2020_CA_{abrev}_csv.zip",
    2025: "https://www.inegi.org.mx/contenidos/programas/eic/2025/microdatos/eic2025_micro_{ent}_csv.zip",
}

columnas = [
    "ent", "mun", "factor", "upm", "estrato",
    "asisten", "conact", "ent_pais_asi", "mun_asi", "ent_pais_trab", "mun_trab",
    "tie_traslado_escu", "tie_traslado_trab",
]


def leer_censo(anio, url):
    """Descarga el zip de INEGI y devuelve la tabla de personas con las columnas del indicador (todo como texto)."""
    with tempfile.TemporaryDirectory(ignore_cleanup_errors=True) as tmp:
        zip_path = os.path.join(tmp, f"{anio}.zip")
        with httpx.stream("GET", url, follow_redirects=True, timeout=8000) as r:
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                for chunk in r.iter_bytes():
                    f.write(chunk)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)

        csvs = [
            os.path.join(root, f)
            for root, _, files in os.walk(tmp)
            for f in files
            if f.lower().endswith(".csv") and not any(k in f.lower() for k in ("diccionario", "descriptor", "catalogo"))
        ]
        personas = [p for p in csvs if "per" in os.path.basename(p).lower()]
        archivo = (personas or csvs)[0]
        print(f"{anio}: {os.path.basename(archivo)}")
        df = pl.read_csv(archivo, infer_schema_length=0, encoding="utf8-lossy")

    df = df.rename({c: c.strip().lower() for c in df.columns})
    df = df.rename({k: v for k, v in {"cve_ent": "ent", "cve_mun": "mun"}.items() if k in df.columns})
    return df.select(columnas)


datos_censo_int = {
    anio: pl.concat([leer_censo(anio, url.format(ent=ent, abrev=abrev_censo2020[ent])) for ent in entidades])
    for anio, url in urls_censo_int.items()
}
for anio, df in datos_censo_int.items():
    print(f"{anio}: {df.height:,} registros de personas (entidades {entidades})")

ZMCM 2020: 63 municipios en entidades ['09', '13', '15']
CVE_ENT
09    16
13     2
15    45
2015: TR_PERSONA09.CSV
2015: TR_PERSONA13.CSV
2015: TR_PERSONA15.CSV
2020: Personas09.CSV
2020: Personas13.CSV
2020: Personas15.CSV
2025: personas09.csv
2025: personas13.csv
2025: personas15.csv
2015: 3,290,517 registros de personas (entidades ['09', '13', '15'])
2020: 2,007,720 registros de personas (entidades ['09', '13', '15'])
2025: 3,718,208 registros de personas (entidades ['09', '13', '15'])


In [4]:
# =============================================================================
# BLOQUE 1: VARIABLES DEL INDICADOR Y DISEÑO MUESTRAL (svy)
# =============================================================================
muestras = {}
cobertura = pl.from_pandas(
    zm.assign(cvegeo=zm["CVE_ENT"].str.zfill(2) + zm["CVE_MUN"].str.zfill(3))[["cvegeo", "NOMGEO"]]
).rename({"NOMGEO": "municipio"})

for anio, df in datos_censo_int.items():
    df = df.with_columns(
        pl.col("factor").cast(pl.Float64),
        pl.col("tie_traslado_trab").str.strip_chars().cast(pl.Int64, strict=False).alias("t_trab"),
        pl.col("tie_traslado_escu").str.strip_chars().cast(pl.Int64, strict=False).alias("t_escu"),
        # Estratos y UPM únicos entre entidades
        (pl.col("ent").str.zfill(2) + "_" + pl.col("estrato")).alias("estrato_ent"),
        (pl.col("ent").str.zfill(2) + "_" + pl.col("estrato") + "_" + pl.col("upm")).alias("upm_anidada"),
        # Residente en un municipio de la ZMCM 2020
        (pl.col("ent").str.zfill(2) + pl.col("mun").str.zfill(3)).is_in(cvegeo_zm).alias("en_zm"),
    ).with_columns(
        # PobTrabEstTraslado: al menos un traslado a estudio o trabajo con tiempo declarado
        (pl.col("t_trab").is_between(1, 5).fill_null(False) | pl.col("t_escu").is_between(1, 5).fill_null(False))
        .alias("se_traslada"),
        # PobTrasladoLargo: algún traslado de más de una hora (cuenta una vez a quien estudia y trabaja)
        (pl.col("t_trab").is_in([4, 5]).fill_null(False) | pl.col("t_escu").is_in([4, 5]).fill_null(False))
        .cast(pl.Int64).alias("traslado_largo"),
    ).with_columns(
        # Dominio de estimación: residentes de la ZMCM que se trasladan
        (pl.col("se_traslada") & pl.col("en_zm")).alias("dominio_zm"),
    )

    # Registros muestrales por municipio de la ZMCM (total y que se trasladan)
    conteo = (
        df.filter(pl.col("en_zm"))
        .group_by(cvegeo=pl.col("ent").str.zfill(2) + pl.col("mun").str.zfill(3))
        .agg(pl.len().alias(f"registros_{anio}"), pl.col("se_traslada").sum().alias(f"se_trasladan_{anio}"))
    )
    cobertura = cobertura.join(conteo, on="cvegeo", how="left")

    diseno = svy.Design(stratum="estrato_ent", psu="upm_anidada", wgt="factor")
    muestras[anio] = svy.Sample(data=df, design=diseno)

    rprint(f"\n[bold]Diseño muestral {anio}[/bold]")
    rprint(muestras[anio])

Diseño muestral 2015

╭────────────── Sample ──────────────╮
│ Survey Data                        │
│   Rows     : 3290517               │
│   Columns  : 24                    │
│   Strata   : 2106                  │
│   PSUs     : 28149                 │
│                                    │
│ Survey Design                      │
│   Row index          svy_row_index │
│   Stratum            estrato_ent   │
│   PSU                upm_anidada   │
│   SSU                None          │
│   Weight             factor        │
│   With replacement   False         │
│   Prob               None          │
│   Hit                None          │
│   MOS                None          │
│   Population size    None          │
│   Replicate weights  None          │
╰────────────────────────────────────╯

Diseño muestral 2020

╭────────────── Sample ──────────────╮
│ Survey Data                        │
│   Rows     : 2007720               │
│   Columns  : 24                    │
│   Strata   : 555                   │
│   PSUs     : 21855                 │
│                                    │
│ Survey Design                      │
│   Row index          svy_row_index │
│   Stratum            estrato_ent   │
│   PSU                upm_anidada   │
│   SSU                None          │
│   Weight             factor        │
│   With replacement   False         │
│   Prob               None          │
│   Hit                None          │
│   MOS                None          │
│   Population size    None          │
│   Replicate weights  None          │
╰────────────────────────────────────╯

Diseño muestral 2025

╭────────────── Sample ──────────────╮
│ Survey Data                        │
│   Rows     : 3718208               │
│   Columns  : 24                    │
│   Strata   : 609                   │
│   PSUs     : 57260                 │
│                                    │
│ Survey Design                      │
│   Row index          svy_row_index │
│   Stratum            estrato_ent   │
│   PSU                upm_anidada   │
│   SSU                None          │
│   Weight             factor        │
│   With replacement   False         │
│   Prob               None          │
│   Hit                None          │
│   MOS                None          │
│   Population size    None          │
│   Replicate weights  None          │
╰────────────────────────────────────╯

In [5]:
# =============================================================================
# BLOQUE 1.1: COBERTURA DE LOS MUNICIPIOS DE LA ZMCM 2020 EN LOS MICRODATOS
# =============================================================================
cobertura = cobertura.sort("cvegeo").fill_null(0)
sin_datos = cobertura.filter(pl.any_horizontal(pl.exclude("cvegeo", "municipio") == 0))
print(f"Municipios ZMCM 2020: {cobertura.height} | con algún año sin registros: {sin_datos.height}")
display(cobertura)

Municipios ZMCM 2020: 63 | con algún año sin registros: 0


cvegeo,municipio,registros_2015,se_trasladan_2015,registros_2020,se_trasladan_2020,registros_2025,se_trasladan_2025
str,str,u32,u32,u32,u32,u32,u32
"""09002""","""Azcapotzalco""",19497,11975,17430,10703,29755,17266
"""09003""","""Coyoacán""",37705,22826,17461,10490,28023,16366
"""09004""","""Cuajimalpa de Morelos""",18712,12205,11713,7403,23899,15698
"""09005""","""Gustavo A. Madero""",64279,38666,19578,12011,33415,20076
"""09006""","""Iztacalco""",19848,12218,19032,11956,32628,19596
"""09007""","""Iztapalapa""",104209,63100,22319,13458,35424,21439
"""09008""","""La Magdalena Contreras""",18182,10966,19280,11177,21224,13193
"""09009""","""Milpa Alta""",19125,11882,9379,6199,24308,16012
"""09010""","""Álvaro Obregón""",43953,26937,18933,11780,21328,12826


In [6]:
# =============================================================================
# BLOQUE 2: ESTIMACIÓN (dominio: residentes ZMCM 2020 que se trasladan) — REPORTES svy
# =============================================================================
resultados = []

for anio, muestra in muestras.items():
    est_prop = muestra.estimation.prop(y="traslado_largo", where=pl.col("dominio_zm"))
    est_total = muestra.estimation.total(y="traslado_largo", where=pl.col("dominio_zm"))

    rprint(f"\n[bold]Estimaciones {anio}[/bold]")
    rprint(est_prop)
    rprint(est_total)

    prop = est_prop.to_polars().filter(pl.col("traslado_largo") == "1")
    total = est_total.to_polars()
    resultados.append({
        "year": anio,
        "personas": total["est"][0],
        "personas_cv_pct": total["cv"][0] * 100,
        "porcentaje": prop["est"][0] * 100,
        "porcentaje_cv_pct": prop["cv"][0] * 100,
        "tipo": "Observado",
    })

tiempos_trabajo = pd.DataFrame(resultados)
display(pl.from_pandas(tiempos_trabajo))

Estimaciones 2015

╭─────────────────── Estimate: PROP (TAYLOR) ───────────────────╮
│ where: dominio_zm                                             │
│                                                               │
│                                                               │
│  traslado_largo      est       se      lci      uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  0                0.8045   0.0014   0.8017   0.8073     0.17  │
│  1                0.1955   0.0014   0.1927   0.1983     0.72  │
│                                                               │
╰───────────────────────────────────────────────────────────────╯

╭──────────────────────── Estimate: TOTAL (TAYLOR) ─────────────────────────╮
│ where: dominio_zm                                                         │
│                                                                           │
│                                                                           │
│             est            se              lci              uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  2,430,764.0000   32,930.2168   2,366,214.1113   2,495,313.8887     1.35  │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

Estimaciones 2020

╭─────────────────── Estimate: PROP (TAYLOR) ───────────────────╮
│ where: dominio_zm                                             │
│                                                               │
│                                                               │
│  traslado_largo      est       se      lci      uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  0                0.7960   0.0021   0.7920   0.8000     0.26  │
│  1                0.2040   0.0021   0.2000   0.2080     1.01  │
│                                                               │
╰───────────────────────────────────────────────────────────────╯

╭──────────────────────── Estimate: TOTAL (TAYLOR) ─────────────────────────╮
│ where: dominio_zm                                                         │
│                                                                           │
│                                                                           │
│             est            se              lci              uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  2,644,478.0000   50,324.4586   2,545,828.4768   2,743,127.5232     1.90  │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

Estimaciones 2025

╭─────────────────── Estimate: PROP (TAYLOR) ───────────────────╮
│ where: dominio_zm                                             │
│                                                               │
│                                                               │
│  traslado_largo      est       se      lci      uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  0                0.7779   0.0018   0.7744   0.7813     0.23  │
│  1                0.2221   0.0018   0.2187   0.2256     0.79  │
│                                                               │
╰───────────────────────────────────────────────────────────────╯

╭──────────────────────── Estimate: TOTAL (TAYLOR) ─────────────────────────╮
│ where: dominio_zm                                                         │
│                                                                           │
│                                                                           │
│             est            se              lci              uci   cv (%)  │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│  2,949,683.0000   41,635.1362   2,868,073.2364   3,031,292.7636     1.41  │
│                                                                           │
╰───────────────────────────────────────────────────────────────────────────╯

year,personas,personas_cv_pct,porcentaje,porcentaje_cv_pct,tipo
i64,f64,f64,f64,f64,str
2015,2.430764e6,1.354727,19.548215,0.719442,"""Observado"""
2020,2.644478e6,1.903002,20.397247,1.010539,"""Observado"""
2025,2.949683e6,1.411512,22.213875,0.792864,"""Observado"""


In [7]:
# =============================================================================
# BLOQUE 3: ESCENARIOS - TIEMPOS DE MOVILIDAD
# =============================================================================
indicador_tiempos = (
    tiempos_trabajo.rename(columns={"year": "anio", "porcentaje": "porcentaje_tiempo_mayor_1h"})
    [["anio", "porcentaje_tiempo_mayor_1h"]]
    .sort_values("anio")
    .reset_index(drop=True)
)

anio_ini = int(indicador_tiempos["anio"].iloc[0])
valor_ini = indicador_tiempos["porcentaje_tiempo_mayor_1h"].iloc[0]
anio_base = int(indicador_tiempos["anio"].max())
linea_base = indicador_tiempos.loc[indicador_tiempos["anio"] == anio_base, "porcentaje_tiempo_mayor_1h"].iloc[0]

tvma = (linea_base / valor_ini) ** (1 / (anio_base - anio_ini)) - 1

multiplicadores = {"inercial": 0.2, "deseable": -1.9, "transformador": -2.66}

anios_proy = list(range(anio_base + 1, 2048))
escenarios = pd.DataFrame({"anio": anios_proy})
for escenario, mult in multiplicadores.items():
    escenarios[escenario] = linea_base * (1 + tvma * mult) ** (escenarios["anio"] - anio_base)

observados = indicador_tiempos.rename(columns={"porcentaje_tiempo_mayor_1h": "observado"})

tabla_final = escenarios.merge(observados, on="anio", how="outer").sort_values("anio")
for escenario in multiplicadores:
    tabla_final[escenario] = tabla_final[escenario].clip(upper=100)
    tabla_final[escenario] = tabla_final["observado"].combine_first(tabla_final[escenario])

tabla_final = pl.from_pandas(
    tabla_final.rename(columns={"anio": "periodo"})
    [["periodo", "inercial", "deseable", "transformador"]]
    .round(2)
    .reset_index(drop=True)
).with_columns(pl.col("periodo").cast(pl.Int64))

print(f"TVMA histórica {anio_ini}-{anio_base}: {tvma:+.4%}")
display(tabla_final)

TVMA histórica 2015-2025: +1.2865%


periodo,inercial,deseable,transformador
i64,f64,f64,f64
2015,19.55,19.55,19.55
2020,20.4,20.4,20.4
2025,22.21,22.21,22.21
2026,22.27,21.67,21.45
2027,22.33,21.14,20.72
2028,22.39,20.62,20.01
2029,22.44,20.12,19.33
2030,22.5,19.63,18.66
2031,22.56,19.15,18.03


In [8]:
# =============================================================================
# BLOQUE 4: EXPORTACIÓN A CSV (UTF-8)
# =============================================================================
archivo_salida = "tardan_mas_1_hora_llegar_zmcm.csv"
tabla_final.to_pandas().to_csv(archivo_salida, index=False, encoding="utf-8")
print(f"Exportado: {archivo_salida} (UTF-8)")

Exportado: tardan_mas_1_hora_llegar_zmcm.csv (UTF-8)
